# 状態空間モデルとTransformer

長い系列を扱うとき、状態空間モデル（SSM）と Transformer は異なるしかたで過去の情報を持ち回ります。ここでは両者の挙動差を小さい系列で確認します。大事なのは数式名を覚えることより、『昔の出来事をどう残して、あとでどう使うか』の違いを掴むことです。


## 長い系列をどう覚えるかで、モデルの性格は変わる

SSM は内部状態を少しずつ更新しながら過去をまとめて持ち、Transformer は必要になったときに過去の位置を見直します。どちらも系列を読む方法ですが、情報の溜め方と取り出し方がかなり違います。ここでいう系列とは、時間順に並んだ数値や単語の列のことです。


ここでは小さな系列に遠く離れたイベントを埋め込み、二つの流儀がその影響をどの程度追えるかを比べます。長距離依存とは、かなり前に起きたことがずっと後の出力へ効いてくる状況です。見たいのは性能順位ではなく、その長い影響をどう扱うかの考え方の違いです。


## 逐次更新と直接参照

SSM は『いま持っている状態に過去を圧縮して持つ』設計で、Transformer は『必要なら過去の特定位置を見直す』設計です。言い換えると、SSM はメモを毎回更新し続ける型、Transformer はあとからノートを開き直す型です。系列長が伸びたときの計算の形も、そこで大きく変わります。


## この notebook の見どころ

`x[5]` と `x[25]` に仕込んだイベントが、SSM 風の更新と Transformer 風の集約でどう扱われるかを MSE と系列の形で見ます。MSE は、予測が目標からどれだけずれたかを平均した量です。ここでは、昔の出来事があとまで残るかどうかを、小さな人工例で目に見える形にします。


ここでの Transformer 側は学習済み attention の完全実装ではなく、考え方を模した簡略代理です。自己注意とは、いまの位置が過去のどこを見るべきかに重みを付ける仕組みです。だから数値の勝敗より、『どの仕組みで遠い情報を拾っているか』に注目してください。


## 比較で読みたいのは性能表ではない

MSE の差だけを見るとランキングの話に流れがちですが、この notebook の主題はそこではありません。どれだけ計算を持ち回るか、遠い出来事をどう残すか、系列が長くなったときにどう振る舞うかを比べるための教材です。ここでいう更新コストとは、新しい 1 点を読むたびにどれくらい計算をやり直す必要があるか、という見方です。


## 読み方の軸

長い系列で効くモデルを一つ選ぶ話ではなく、『内部状態に蓄えるか、必要な場所をあとで見直すか』という二つの発想を整理するつもりで進めてください。


## 長距離依存を含む系列を作る

まずは遠く離れた位置のイベントが後段に効くような系列を用意し、比較の土台を作ります。前の出来事があとまで尾を引くような並びを意図的に作る、ということです。


In [ ]:
import numpy as np
np.random.seed(2)

T = 40
x = np.zeros(T)
x[5] = 1.0
x[25] = 0.8  # 長距離イベント

y_target = np.zeros(T)
for t in range(1, T):
    y_target[t] = 0.7 * y_target[t - 1] + x[t]


## 二つの集約のしかたを並べる

次に、逐次更新型と注意型で同じ系列を読み、どこで情報の拾い方が変わるかを見ます。片方は毎回メモを更新し、もう片方は必要な過去へ重み付きで目を向けます。


In [ ]:
# SSM: h_t = a h_{t-1} + b x_t, y_t = c h_t
a, b, c = 0.8, 1.0, 1.0
h = 0.0
y_ssm = np.zeros(T)
for t in range(T):
    h = a * h + b * x[t]
    y_ssm[t] = c * h

# Transformer風: 位置に依存した自己注意で過去を加重平均
idx = np.arange(T)
y_attn = np.zeros(T)
for t in range(T):
    past = x[: t + 1]
    dist = (t - idx[: t + 1]).astype(float)
    score = -0.15 * dist  # 近いほど重い単純な注意
    w = np.exp(score - score.max())
    w /= w.sum()
    y_attn[t] = np.dot(w, past)

mse_ssm = np.mean((y_ssm - y_target) ** 2)
mse_attn = np.mean((y_attn - y_target) ** 2)
print('SSM-like MSE        =', round(mse_ssm, 6))
print('Transformer-like MSE=', round(mse_attn, 6))


SSM は更新コストの軽さに、Transformer は必要な位置を直接見にいける分かりやすさに強みがあります。この notebook の比較は、その設計差を手触りとして掴むためのものです。どちらが常に上かではなく、どういう覚え方を選んでいるかを見るノートだと捉えてください。
